# ♟️ Chess Numba — Phase 8 Colab SPSA & Benchmark Runner
## 空步剪枝 (NMP) 體系 SF19 結構對齊、零調參前測與連續 SPSA

本 Notebook 執行 **Phase 8**：
1. **【Part 1: 零調參前測】**：以目前 HCE 預設常數，讓對齊 SF19 Step 10 新演算法的引擎直接對決舊引擎（`chess_engine.classical_old`），量測純演算法邏輯改善效果。
2. **【Part 2: 4 軸連續 SPSA】**：在 Colab（7 併發、`inprocess` 單次 JIT）高速搜尋 NMP 在 HCE 空間的最佳連續參數。
3. **【Part 3: 調參後終極決戰】**：套用 SPSA 最佳候選參數，與舊引擎進行長時控終極驗收（300,000 nodes）。

技術計畫書：`chess_engine/classical/PHASE8_NMP_SF19_ALIGNMENT_PLAN.md`

### 🎯 4 個連續 SPSA 調參軸（全為 `SearchContext.tune[]` 運行時槽位，免重編譯）
| 參數名稱 | 基準值 | 探索範圍 | 步長 c | tune 槽位 | 物理意義 |
| :-- | --: | :-: | --: | --: | :-- |
| `NMP_BASE` | 200 | [100, 350] | 12 | 26 | 靜態基礎門檻（控制 NMP 整體鬆緊尺度） |
| `NMP_DEPTH_COEF` | 14 | [0, 30] | 2 | 27 | 隨深度放寬門檻之斜率 |
| `NMP_IMP_COEF` | 45 | [0, 90] | 5 | 28 | 局勢改善時之放寬補貼（防守反擊保護） |
| `NMP_EVAL_MARGIN_DIV` | 150 | [64, 400] | 16 | 20 | 超額靜態估值換取動態減深 $R$ 之除數 |

> **注意**：舊引擎 `chess_engine.classical_old` 保持為對照基線（未包含 Phase 8 改動），請勿同步以免失去對照基準。


In [ ]:
# 1. 檢視 Colab 硬體
!lscpu | grep "Model name\|Socket\|Thread\|NUMA\|CPU(s)"
!python3 --version

In [ ]:
# 2. 設定（續跑時請勿更動 PARAMS / GAMES / NODES / CONCURRENCY / TOTAL_ITERS / SPSA_A_RATIO）
REPO_URL = "https://github.com/b08203031/chess_numba.git"
BRANCH = "feat-initial-bitboard-setup"
RUN_TAG = "nmp_p8_l1"

PARAMS = [
    "NMP_BASE",
    "NMP_DEPTH_COEF",
    "NMP_IMP_COEF",
    "NMP_EVAL_MARGIN_DIV",
]

# ---- Part 1: 零調參前測 (vs classical_old) ----
PRE_NODES = 200000
PRE_GAMES = 200
PRE_TT_MB = 64

# ---- Part 2: L1 SPSA 雲端調參 ----
NODES = 200000            # 200k nodes 兼顧搜尋深度與每輪速度
GAMES = 200              # 每輪 plus/minus 對局數
TOTAL_ITERS = 30        # 計畫輪數
CHUNK_ITERS = TOTAL_ITERS
TT_MB = 32
SPSA_A_RATIO = 0.2
CONCURRENCY = 7          # 8 vCPU -> 7 併發
TAIL = 0.3               # Polyak-Ruppert 尾端比例

# ---- Part 3: 調參後終極決戰 (vs classical_old) ----
L2_NODES = 300000
L2_GAMES = 400
L2_TT_MB = 64

USE_DRIVE = True         # 狀態/日誌/候選存到 Google Drive
RUN_SMOKE = True         # 第一次先跑 8 局冒煙（驗證指令與 JIT）

In [ ]:
# 3. 持久化目錄（Google Drive）
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = "/content/drive/MyDrive/chess_numba_p8"
else:
    PERSIST = "/content/chess_numba_p8_persist"

for sub in ("states", "logs", "results"):
    os.makedirs(f"{PERSIST}/{sub}", exist_ok=True)

STATE_FILE = f"{PERSIST}/states/{RUN_TAG}.json"
LOG_FILE = f"{PERSIST}/logs/{RUN_TAG}.jsonl"
CAND_UNIFORM = f"{PERSIST}/results/candidate_p8_tail_uniform.json"
CAND_LINEAR = f"{PERSIST}/results/candidate_p8_tail_linear.json"
print("persist dir:", PERSIST)
print("state:", STATE_FILE)
print("log  :", LOG_FILE)

In [ ]:
# 4. 取得或更新程式碼倉庫
import os
if not os.path.exists("/content/chess_numba"):
    !git clone -b {BRANCH} {REPO_URL} /content/chess_numba
else:
    !cd /content/chess_numba && git fetch && git checkout {BRANCH} && git pull

%cd /content/chess_numba
!git log -1 --oneline

In [ ]:
# 5. 驗證 Phase 8 初始基線常數
import sys, json
sys.path.insert(0, "/content/chess_numba")
from chess_engine.classical import constants as C
from tune_search.search_param_registry import get_default_params, get_spec

assert C.TUNE_SIZE == 42, f"TUNE_SIZE={C.TUNE_SIZE}; 需為 42"

expected_phase8_defaults = {
    "NMP_BASE": 200,
    "NMP_DEPTH_COEF": 14,
    "NMP_IMP_COEF": 45,
    "NMP_EVAL_MARGIN_DIV": 150,
}
baseline = get_default_params(PARAMS)
assert baseline == expected_phase8_defaults, f"基線與 Phase 8 預期不一致！{baseline} vs {expected_phase8_defaults}"

BASELINE_JSON = "/content/chess_numba/tune_search/baseline_nmp_p8.json"
with open(BASELINE_JSON, "w", encoding="utf-8") as f:
    json.dump(baseline, f, indent=2)

print(f"{'param':30s} {'default':>8s} {'range':>14s} {'c':>5s} {'slot':>5s}")
for name in PARAMS:
    s = get_spec(name)
    print(f"{name:30s} {s.default:8d} {'['+str(s.minimum)+','+str(s.maximum)+']':>14s} {s.step:5d} {s.runtime_index:5d}")
print("✅ Phase 8 基線驗證通過；基線檔:", BASELINE_JSON)

## ⚔️ Part 1: 重構後零調參直接對戰舊引擎 (Pre-Tuning Battle vs `classical_old`)

在此步驟，我們**完全不調整任何參數**（維持原有 200, 14, 45, 150），直接以新版演算法（`nmp_min_ply` 局部解禁 + `!is_win` 假勝局防護 + `cutNode=false` 驗證搜尋 + 解除 eval_margin clamp）與舊引擎 `classical_old` 對決。

* 若直接獲勝：證明 SF19 演算法結構自身即帶來明確的正向 Elo！
* 若持平或微負：亦屬正常，因為舊參數是在舊架構上調出的，此時正好進入 Part 2 透過 SPSA 尋找新結構的最優解！

In [ ]:
# 6. Part 1 零調參對戰 (200 局 @ 200k nodes vs classical_old)
!python tools/tournament.py \
  --engine1-module chess_engine.classical \
  --engine2-module chess_engine.classical_old \
  --name1 "New_SF19_NMP_PreTune" \
  --name2 "Old_Baseline" \
  --nodes {PRE_NODES} \
  --games {PRE_GAMES} \
  --concurrency {CONCURRENCY} \
  --tt-mb {PRE_TT_MB} \
  --backend inprocess \
  --pgn "{PERSIST}/results/pre_tune_match_vs_old.pgn"

## 🔬 Part 2: 雲端 SPSA 4 軸高速調參 (L1 SPSA 4-Axis Continuous Tuning)

在 Part 1 掌握演算法基線表現後，此處透過 SPSA 對 4 個連續門檻進行梯度探索，解開 HCE 尺度下的最優空步剪枝超參數。

In [ ]:
# 7. SPSA 冒煙測試（8 局 @ 20k nodes；確認 inprocess worker 與 JIT 通路）
if RUN_SMOKE:
    !python -m tune_search.search_tuner \
      --params {",".join(PARAMS)} \
      --backend inprocess --scale-mode integer \
      --iter 1 --games 8 --nodes 20000 \
      --spsa-a-ratio {SPSA_A_RATIO} --concurrency {CONCURRENCY} \
      --adjudicate --log /tmp/p8_smoke.jsonl --state ''
else:
    print("RUN_SMOKE=False，略過。")

In [ ]:
# 8. 執行 L1 SPSA（支援斷線自動 --resume）
import json
done_iters = 0
if os.path.exists(STATE_FILE):
    try:
        done_iters = json.load(open(STATE_FILE)).get("iter_count", 0)
    except Exception:
        pass

remain = max(0, TOTAL_ITERS - done_iters)
run_now = min(CHUNK_ITERS, remain)
print(f"已完成 {done_iters}/{TOTAL_ITERS} 輪；本次執行 {run_now} 輪")

if run_now > 0:
    !python -m tune_search.search_tuner \
      --params {",".join(PARAMS)} \
      --backend inprocess --scale-mode integer \
      --iter {run_now} --games {GAMES} --nodes {NODES} \
      --spsa-a-ratio {SPSA_A_RATIO} --concurrency {CONCURRENCY} \
      --tt-mb {TT_MB} --adjudicate \
      --resume {STATE_FILE} --log {LOG_FILE}
else:
    print("已達總輪數，無需執行。")

In [ ]:
# 9. 產出候選參數（Polyak-Ruppert 尾端平滑抽取）
!python tune_search/summarize_run.py {LOG_FILE} --tail {TAIL} --weight-mode uniform --out {CAND_UNIFORM}
!python tune_search/summarize_run.py {LOG_FILE} --tail {TAIL} --weight-mode linear  --out {CAND_LINEAR}
print("\n=== uniform candidate ===")
!cat {CAND_UNIFORM}
print("\n=== linear candidate ===")
!cat {CAND_LINEAR}

In [ ]:
# 10. 候選合理性審查與漂移監控
import json
if os.path.exists(CAND_UNIFORM):
    cand = json.load(open(CAND_UNIFORM, encoding="utf-8"))["averaged_params"]
    print(f"{'param':30s} {'baseline':>9s} {'candidate':>10s} {'delta%':>8s}  note")
    for name in PARAMS:
        s = get_spec(name)
        v = int(cand[name]); b = expected_phase8_defaults[name]
        notes = []
        if v <= s.minimum or v >= s.maximum:
            notes.append("⚠ 撞到範圍邊界")
        print(f"{name:30s} {b:9d} {v:10d} {100.0*(v-b)/b:+8.1f}  {' '.join(notes)}")

## 🏆 Part 3: 調參後終極決戰 vs 舊引擎 (Post-Tuning Final Battle vs `classical_old`)

在此步驟，我們將 SPSA 產出的最佳候選向量套入現行引擎，並在 **長時控（300,000 nodes, 64MB TT, 400 局）** 下與基準引擎 `chess_engine.classical_old` 進行決戰驗收！

In [ ]:
# 11. 將候選參數注入現行引擎環境，展開 Part 3 終極對戰
import json
cand_params = json.load(open(CAND_UNIFORM, encoding="utf-8"))["averaged_params"]
print("套用候選參數:", cand_params)

# 透過 --engine1-env 或專屬對抗模式與 classical_old 對抗
!python -m tune_search.search_param_match \
  --candidate {CAND_UNIFORM} \
  --baseline {BASELINE_JSON} \
  --nodes {L2_NODES} \
  --games {L2_GAMES} \
  --concurrency {CONCURRENCY} \
  --tt-mb {L2_TT_MB} \
  --sprt 0.0,3.0 \
  --backend inprocess

In [ ]:
# 12. （附加對照）候選版與未調參版直接 A/B 比較驗收
!python tools/tournament.py \
  --engine1-module chess_engine.classical \
  --engine2-module chess_engine.classical_old \
  --name1 "New_SF19_NMP_Tuned" \
  --name2 "Old_Baseline" \
  --nodes {L2_NODES} \
  --games {L2_GAMES} \
  --concurrency {CONCURRENCY} \
  --tt-mb {L2_TT_MB} \
  --backend inprocess \
  --pgn "{PERSIST}/results/final_post_tune_vs_old.pgn"

### 🏁 收官與落地步驟
1. 下載 Drive 中的 `candidate_p8_tail_uniform.json` 與對戰結果日誌。
2. 修改 `chess_engine/classical/constants.py` 的 `NMP_BASE` / `NMP_DEPTH_COEF` / `NMP_IMP_COEF` / `NMP_EVAL_MARGIN_DIV`。
3. 執行 `python -m tuner.sync_constants_snapshot`。
4. 更新 `SEARCH_ANALYSIS.md` 與 `PHASE8_NMP_SF19_ALIGNMENT_PLAN.md` 戰役章節。
5. 驗收完成後再使用 `python tools/sync_classical_old.py --sync` 同步舊基準。